# 🩹 Clase 03 · Práctica guiada: datos faltantes y métodos de imputación
Dataset: **viviendas en Lima** (sintético, con problemas de calidad puestos a propósito).

El profesor dio una clase **teórica**: de dónde salen los faltantes (errores de programación,
datos no reportados, manejo inapropiado), la "sexta categoría" *blanco / no reportó*, el **sesgo**
y la anécdota del **LEFT JOIN** que fabrica nulos. Aquí llevamos cada idea a código.

> Los métodos concretos (moda, constante, mediana por grupo) son 📌 Extra: el profe los profundiza en clases siguientes.

In [ ]:
import sys
from pathlib import Path

# Localiza la raíz del repo, para que el notebook funcione desde cualquier carpeta
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "framework").exists())
sys.path.insert(0, str(RAIZ))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from framework.testing import ruta_dataset  # si el CSV no existe, lo genera

df = pd.read_csv(ruta_dataset("viviendas_lima.csv"))
df.shape

## Paso 0 · Repaso: ¿dónde faltan datos?
`df.isna().sum()` cuenta los NaN por columna (clases 01 y 02).

In [ ]:
faltantes = df.isna().sum()
faltantes[faltantes > 0].sort_values(ascending=False)

## Paso 1 · La "sexta categoría": vacíos disfrazados
El profesor: *"tengo una variable categórica con cinco categorías, pero me aparece una sexta que dice blanco, vacío, no reportó"*.

Para pandas, el texto `"N/A"` o `" "` es **un valor más**, no un hueco. Ojo con un detalle:
`read_csv` ya convierte algunos textos (`"N/A"`, `"NA"`, `""`...) a NaN **automáticamente**.
Para ver el problema "crudo", leemos `cochera` desactivando esa conversión.

In [ ]:
cruda = pd.read_csv(ruta_dataset("viviendas_lima.csv"), usecols=["id", "cochera"],
                    keep_default_na=False, dtype={"cochera": str})
print("NaN según pandas:", cruda["cochera"].isna().sum())
cruda["cochera"].value_counts()

`isna()` dice **0** faltantes, pero hay 18 `"N/A"`: un hueco disfrazado. (Y de paso, `"si"`, `"SI"`, `"no "`
son la misma categoría mal escrita: eso es limpieza de texto, lo veremos con encoding.)

La receta: normalizar el texto (`strip` + `lower`) **solo para comparar**, y marcar como NaN lo que esté en una lista de marcadores.

In [ ]:
marcadores = ["", "n/a", "na", "-", "null", "none", "sin dato", "no reporto"]

texto = cruda["cochera"].str.strip().str.lower()   # "  N/A " -> "n/a"
mascara = texto.isin(marcadores)
print("Vacíos disfrazados encontrados:", mascara.sum())

limpia = cruda.copy()
limpia.loc[mascara, "cochera"] = np.nan
print("NaN después de limpiar:", limpia["cochera"].isna().sum())

### ✏️ Tu turno 1
Construye un DataFrame pequeño con una columna `telefono` que tenga: un número válido, `"-"`, `"  "`, `"No reporto"`
y `None` (el caso del adulto mayor que no digita su teléfono). Aplica la receta y cuenta cuántos NaN quedan. ¿Cuántos esperabas?

In [ ]:
# tu código aquí

## Paso 2 · Por qué importa: los modelos no aceptan NaN
El profesor: *"muchos modelos de aprendizaje no aceptan conjuntos de datos con valores en falta"*.
Mira lo que pasa si le das NaN a scikit-learn:

In [ ]:
from sklearn.linear_model import LinearRegression

X = df[["area_m2", "antiguedad_anios"]]
y = df["precio_usd"]
try:
    LinearRegression().fit(X, y)
except ValueError as e:
    print("Error:", str(e)[:120], "...")

Dos salidas fáciles... y peligrosas:
- **Borrar filas** con `dropna()`: pierdes datos (y si el faltante es MAR, pierdes *siempre el mismo tipo* de fila → sesgo).
- **Rellenar** (imputar): hay que elegir *con qué*. Eso es lo que sigue.

In [ ]:
print("Filas totales:", len(df))
print("Filas si borro TODO lo que tenga algún NaN:", len(df.dropna()))

## Paso 3 · 📌 Extra: imputar una categórica (moda vs. "Desconocido")
`estado` tiene 73 huecos. Dos opciones clásicas:

In [ ]:
moda = df["estado"].mode().iloc[0]
print("Moda de estado:", moda)

comparacion = pd.DataFrame({
    "original": df["estado"].value_counts(dropna=False),
    "con_moda": df["estado"].fillna(moda).value_counts(),
    "con_constante": df["estado"].fillna("Desconocido").value_counts(),
})
comparacion

- Con la **moda**, "Bueno" se infla de 1071 a 1144 y **se pierde la señal** de que el dato faltaba.
- Con **"Desconocido"**, el hueco se vuelve una categoría explícita: el modelo puede aprender si "no reportó" significa algo
  (los que no reportan algo no siempre son una muestra al azar).

## Paso 4 · 📌 Extra: imputar una numérica por grupo
En la clase 02 vimos que `distancia_centro_km` falta más en distritos **periféricos** (MAR).
¿Qué pasa si rellenamos con la mediana **global**?

In [ ]:
mediana_global = df["distancia_centro_km"].median()
mediana_por_distrito = df.groupby("distrito")["distancia_centro_km"].median().sort_values()
print(f"Mediana global: {mediana_global:.2f} km")
mediana_por_distrito

A una vivienda de Ate (~14.8 km) le pondríamos ~10 km: la "acercamos" al centro → **sesgo**.
La clave es `groupby(...).transform("median")`: devuelve **una mediana por fila** (la de su grupo), del mismo largo que `df`.

In [ ]:
faltaba = df["distancia_centro_km"].isna()
mediana_grupo = df.groupby("distrito")["distancia_centro_km"].transform("median")

global_ = df["distancia_centro_km"].fillna(mediana_global)
por_grupo = df["distancia_centro_km"].fillna(mediana_grupo)

pd.DataFrame({
    "distrito": df.loc[faltaba, "distrito"],
    "imputado_global": global_[faltaba],
    "imputado_por_grupo": por_grupo[faltaba],
}).head(8)

### ✏️ Tu turno 2
Imputa `antiguedad_anios` con la mediana **por `tipo_inmueble`**. Compara las medianas de cada tipo:
¿cambia mucho respecto a usar la global? (Pista: en la clase 02 vimos que este faltante era MCAR.)

In [ ]:
# tu código aquí

## Paso 5 · La anécdota del LEFT JOIN: nulos fabricados por el script
El profesor: *"en vez de un INNER JOIN hizo un LEFT JOIN [...] me sacaste nulos"*.

Simulemos una tabla de distritos donde alguien **olvidó cargar** SJL y Ate:

In [ ]:
conocidos = sorted(set(df["distrito"]) - {"SJL", "Ate"})
info_distritos = pd.DataFrame({"distrito": conocidos, "codigo_ubigeo": range(100, 100 + len(conocidos))})

inner = pd.merge(df, info_distritos, on="distrito", how="inner")
left = pd.merge(df, info_distritos, on="distrito", how="left", indicator=True)

print("Filas originales:", len(df))
print("INNER JOIN:", len(inner), "filas (se perdieron filas sin avisar)")
print("LEFT JOIN :", len(left), "filas")
print("NaN en codigo_ubigeo tras el LEFT:", left["codigo_ubigeo"].isna().sum())
left["_merge"].value_counts()

Esos 292 NaN **no son clientes que no reportaron**: los fabricó la consulta. Si no lo notas, te pasas
"todo un día con dolor de cabeza" imputando algo que había que arreglar en el SQL.

💡 `indicator=True` agrega `_merge` (`both` / `left_only` / `right_only`): es tu detector de filas sin pareja.
Antes de imputar, pregúntate: **¿este nulo viene del dato o del JOIN?**

## Paso 6 · Imputar no es gratis: medir el sesgo
Si rellenas muchos huecos con el mismo número, la variable se **aplana** (baja la desviación estándar).
Si rellenas con un número absurdo (p. ej. 0), además **mueves la media**.

In [ ]:
ant = df["antiguedad_anios"]
opciones = {
    "original (sin NaN)": ant.dropna(),
    "mediana": ant.fillna(ant.median()),
    "cero": ant.fillna(0),
}
pd.DataFrame({k: {"media": v.mean(), "std": v.std()} for k, v in opciones.items()}).T.round(2)

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for eje, (nombre, serie) in zip(ejes, opciones.items()):
    serie[serie <= 60].plot.hist(bins=30, ax=eje, title=nombre)
plt.tight_layout()

Mira los picos: con la mediana aparece una barra artificial en 10 años; con cero, una en 0.
Un modelo entrenado con eso aprende un comportamiento que **no es el normal** de los datos → peor **generalización**.

### ✏️ Tu turno 3
Calcula el cambio porcentual de la std, `(std_despues - std_antes) / std_antes * 100`, para la imputación con mediana
y con cero. ¿Cuál "deforma" más la variable? ¿Qué pasaría si en vez de 170 huecos hubiera 1000?

In [ ]:
# tu código aquí

## Resumen
| Idea del profesor | En pandas | Función del ejercicio |
|---|---|---|
| La 6.ª categoría "blanco / no reportó" | `.str.strip().str.lower().isin(marcadores)` | `normalizar_vacios` |
| Completar lo que falta (categórica) | `fillna(moda)` / `fillna("Desconocido")` | `imputar_categorica` |
| Evitar el sesgo al imputar | `groupby(g)[c].transform("median")` | `imputar_mediana_por_grupo` |
| El culpable era el script SQL | `merge(how="left", indicator=True)` | `nulos_por_join` |
| Sesgo / generalización | comparar `mean()` y `std()` antes y después | `impacto_imputacion` |